# Archived  Analysis

This was a purely rule-based system for counting squat repetitions and providing real-time feedback.



In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
import time

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

def calculate_angle(a, b, c):
    a = np.array(a); b = np.array(b); c = np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    if angle > 180.0: angle = 360 - angle
    return angle

class ManualValidator:
    def __init__(self, video_paths):
        self.video_paths = video_paths
        self.current_video_idx = 0
        self.all_reps = [] 
        self.current_rep_idx = 0
        self.correct_count = 0
        self.total_count = 0

        self.image_widget = widgets.Image(format='jpeg', width=640)
        self.play = widgets.Play(value=0, min=0, max=100, step=1, interval=100)
        self.slider = widgets.IntSlider(value=0, min=0, max=100, step=1, continuous_update=False)
        widgets.jslink((self.play, 'value'), (self.slider, 'value'))
        self.slider.observe(self._update_image, names='value')
        
        self.info_html = widgets.HTML()
        self.correct_button = widgets.Button(description="✅ Correct", button_style='success')
        self.incorrect_button = widgets.Button(description="❌ Incorrect", button_style='danger')
        self.correct_button.on_click(self._record_correct)
        self.incorrect_button.on_click(self._record_incorrect)
        
        self.output = widgets.VBox([
            self.info_html,
            self.image_widget,
            widgets.HBox([self.play, self.slider]),
            widgets.HBox([self.correct_button, self.incorrect_button])
        ])

    def _update_image(self, change):
        frame_idx = change['new']
        _, rep_label, rep_frames = self.all_reps[self.current_rep_idx]
        if 0 <= frame_idx < len(rep_frames):
            img = Image.fromarray(rep_frames[frame_idx])
            self.image_widget.value = self._image_to_byte_array(img)

    def _image_to_byte_array(self, image: Image) -> bytes:
        import io
        img_byte_arr = io.BytesIO()
        image.save(img_byte_arr, format='JPEG')
        return img_byte_arr.getvalue()

    def _extract_reps(self):
        """Pre-processes the current video to find all rep segments first."""
        video_path = self.video_paths[self.current_video_idx]
        print(f"Analyzing video: {video_path.name}...")
        
        cap = cv2.VideoCapture(str(video_path))
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()

        reps_found = []
        counter, stage = 0, None
        rep_frames_buffer = []

        with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
            for i, frame in enumerate(frames):
                results = pose.process(frame)
                
                annotated_frame = frame.copy()
                if results.pose_landmarks:
                    mp_drawing.draw_landmarks(annotated_frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2), 
                                            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
                rep_frames_buffer.append(annotated_frame)
                
                try:
                    landmarks = results.pose_landmarks.landmark
                    hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
                    knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
                    ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]
                    angle = calculate_angle(hip, knee, ankle)
                    
                    if angle > 160:
                        if stage == 'down':
                            min_angle_in_rep = min(angles_in_rep)
                            if min_angle_in_rep >= 90: system_label = "Poor (Too High)"
                            elif min_angle_in_rep < 90: system_label = "Too Low"
                            reps_found.append((system_label, rep_frames_buffer))
                        rep_frames_buffer = [annotated_frame]
                        angles_in_rep = []
                        stage = "up"
                        
                    if stage == 'up':
                        angles_in_rep.append(angle)

                    if angle < 90 and stage =='up':
                        stage="down"
                except:
                    if stage == 'up':
                        min_angle_in_rep = min(angles_in_rep) if angles_in_rep else 180
                        if min_angle_in_rep >= 90: system_label = "Poor (Too High)"
                        else: system_label = "Good" # Assume good if they broke 90
                        reps_found.append((system_label, rep_frames_buffer))
                        rep_frames_buffer = []
                        stage = None


    
        final_reps = []
        for label, frames in reps_found:
             min_angle = 180
             with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
                for frame in frames:
                     results = pose.process(frame)
                     if results.pose_landmarks:
                        try:
                            landmarks = results.pose_landmarks.landmark
                            hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
                            knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
                            ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]
                            angle = calculate_angle(hip, knee, ankle)
                            min_angle = min(min_angle, angle)
                        except: continue
             final_label = "Good" if 90 <= min_angle <= 160 else ("Too Low" if min_angle < 90 else "Poor (Too High)")
             final_reps.append((len(final_reps)+1, final_label, frames))
        self.all_reps = final_reps
        self.current_rep_idx = 0
        print(f"Found {len(self.all_reps)} reps to validate.")

    def _show_next_rep(self):
        if self.current_rep_idx >= len(self.all_reps):
            self._finish_video()
            return

        rep_num, rep_label, rep_frames = self.all_reps[self.current_rep_idx]
        self.info_html.value = f"<h3>Video {self.current_video_idx+1}/{len(self.video_paths)}: Rep #{rep_num}</h3>" \
                               f"<b>System's Label:</b> <span style='color: blue; font-size: 1.2em;'>{rep_label}</span>"
        
        self.slider.max = len(rep_frames) - 1
        self.play.max = len(rep_frames) - 1
        self.slider.value = 0
        self.play.value = 0
        self._update_image({'new': 0})
        self.correct_button.disabled = False
        self.incorrect_button.disabled = False

    def _record_and_advance(self, is_correct):
        self.correct_button.disabled = True
        self.incorrect_button.disabled = True
        self.total_count += 1
        if is_correct:
            self.correct_count += 1
        
        self.current_rep_idx += 1
        self._show_next_rep()

    def _record_correct(self, b): self._record_and_advance(True)
    def _record_incorrect(self, b): self._record_and_advance(False)

    def _finish_video(self):
        self.current_video_idx += 1
        if self.current_video_idx < len(self.video_paths):
            self.start() 
        else:
            clear_output(wait=True)
            accuracy = (self.correct_count / self.total_count * 100) if self.total_count > 0 else 0
            display(widgets.HTML(f"<h1>Validation Complete!</h1>" \
                                 f"<h2>Final Accuracy: {self.correct_count}/{self.total_count} ({accuracy:.2f}%)</h2>"))

    def start(self):
        if self.current_video_idx >= len(self.video_paths):
            print("All videos processed.")
            return
            
        display(self.output)
        self._extract_reps()
        self._show_next_rep()

import os
import kagglehub

TEST_VIDEO_DIR = Path('test_video')
YOUTUBE_PATH = Path('data/youtube_videos')
kaggle_path = kagglehub.dataset_download("hasyimabdillah/workoutfitness-video")
KAGGLE_PATH = Path(kaggle_path)

VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv'}

test_videos = [p for p in TEST_VIDEO_DIR.rglob('*') if p.suffix.lower() in VIDEO_EXTS]
youtube_videos = [p for p in YOUTUBE_PATH.rglob('*') if p.suffix.lower() in VIDEO_EXTS]
kaggle_videos = [p for p in KAGGLE_PATH.rglob('*') if 'squat' in p.name.lower() and p.suffix.lower() in VIDEO_EXTS]

all_videos_to_validate = test_videos + youtube_videos + kaggle_videos

print(f"Found {len(all_videos_to_validate)} total videos to validate from all sources.")

if all_videos_to_validate:
    validator = ManualValidator(all_videos_to_validate)
    validator.start()
else:
    print("No videos found. Please check your data paths.")



Found 99 total videos to validate from all sources.


Analyzing video: squat_8.mp4...


I0000 00:00:1755188648.432280 5767562 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
W0000 00:00:1755188648.525872 5778792 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1755188648.543959 5778792 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1755188650.386394 5767562 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M1
W0000 00:00:1755188650.459358 5778820 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1755188650.476252 5778822 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Found 1 reps to validate.
